# Deep Learning / NLP Text Classifier with PyTorch
### Sentiment Analysis: Positive vs. Negative

**Project objective:** Build an end-to-end NLP pipeline that converts text into TF-IDF vectors, trains a multi-layer neural network, evaluates it on unseen text, plots training/validation loss and accuracy, and reports prediction confidence.

**Model requirements covered**
- TF-IDF text vectorization
- Multi-layer PyTorch neural network
- Dropout and Batch Normalization
- Early stopping based on validation loss
- Training/validation convergence curves
- Test-set evaluation and sample inference with confidence scores

> This project includes a small, self-contained demonstration dataset (`sentiment_dataset.csv`) so it can run without downloading a dataset. For a stronger research result, replace it with a larger labelled sentiment dataset and keep the same pipeline.

## 1. Install / import dependencies

In [ ]:
# If needed, uncomment this line in a fresh Jupyter environment:
# %pip install numpy pandas matplotlib scikit-learn torch

import os
import random
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

import torch
import torch.nn as nn
from torch.utils.data import TensorDataset, DataLoader

from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics import (
    accuracy_score, classification_report, confusion_matrix,
    ConfusionMatrixDisplay
)

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("PyTorch:", torch.__version__)
print("Training device:", DEVICE)

## 2. Load and inspect the dataset

In [ ]:
# The CSV is stored beside this notebook in the project folder.
data_path = "sentiment_dataset.csv"
if not os.path.exists(data_path):
    raise FileNotFoundError(
        "sentiment_dataset.csv was not found. Keep the CSV in the same folder as this notebook."
    )

df = pd.read_csv(data_path)
df = df.dropna(subset=["text", "label"]).drop_duplicates().reset_index(drop=True)

print("Dataset shape:", df.shape)
display(df.head())
print("\\nClass distribution:")
print(df["label"].value_counts())

## 3. Encode labels and split the data

In [ ]:
label_to_id = {"negative": 0, "positive": 1}
id_to_label = {v: k for k, v in label_to_id.items()}
y = df["label"].map(label_to_id).to_numpy()
texts = df["text"].astype(str).to_numpy()

X_train_text, X_temp_text, y_train, y_temp = train_test_split(
    texts, y, test_size=0.30, random_state=SEED, stratify=y
)
X_val_text, X_test_text, y_val, y_test = train_test_split(
    X_temp_text, y_temp, test_size=0.50, random_state=SEED, stratify=y_temp
)

print(f"Train: {len(y_train)} | Validation: {len(y_val)} | Test: {len(y_test)}")

## 4. Tokenize and vectorize using TF-IDF

In [ ]:
# Fit TF-IDF only on training text to avoid data leakage.
vectorizer = TfidfVectorizer(
    lowercase=True,
    stop_words="english",
    ngram_range=(1, 2),
    max_features=500,
    sublinear_tf=True
)

X_train = vectorizer.fit_transform(X_train_text).toarray().astype(np.float32)
X_val = vectorizer.transform(X_val_text).toarray().astype(np.float32)
X_test = vectorizer.transform(X_test_text).toarray().astype(np.float32)

print("TF-IDF feature count:", X_train.shape[1])

## 5. Convert data to PyTorch tensors

In [ ]:
def make_loader(features, labels, batch_size=8, shuffle=False):
    x_tensor = torch.tensor(features, dtype=torch.float32)
    y_tensor = torch.tensor(labels, dtype=torch.long)
    dataset = TensorDataset(x_tensor, y_tensor)
    return DataLoader(dataset, batch_size=batch_size, shuffle=shuffle)

train_loader = make_loader(X_train, y_train, shuffle=True)
val_loader = make_loader(X_val, y_val)
test_loader = make_loader(X_test, y_test)

input_dim = X_train.shape[1]
num_classes = 2

## 6. Build the multi-layer neural network

In [ ]:
class SentimentClassifier(nn.Module):
    def __init__(self, input_dim, num_classes=2):
        super().__init__()
        self.network = nn.Sequential(
            nn.Linear(input_dim, 128),
            nn.BatchNorm1d(128),
            nn.ReLU(),
            nn.Dropout(0.35),

            nn.Linear(128, 64),
            nn.BatchNorm1d(64),
            nn.ReLU(),
            nn.Dropout(0.25),

            nn.Linear(64, num_classes)
        )

    def forward(self, x):
        return self.network(x)

model = SentimentClassifier(input_dim, num_classes).to(DEVICE)
criterion = nn.CrossEntropyLoss()
optimizer = torch.optim.Adam(model.parameters(), lr=0.001, weight_decay=1e-4)

print(model)

## 7. Train the model with early stopping

In [ ]:
def run_epoch(loader, training=False):
    model.train() if training else model.eval()
    total_loss, all_preds, all_targets = 0.0, [], []

    for xb, yb in loader:
        xb, yb = xb.to(DEVICE), yb.to(DEVICE)

        if training:
            optimizer.zero_grad()

        with torch.set_grad_enabled(training):
            logits = model(xb)
            loss = criterion(logits, yb)
            if training:
                loss.backward()
                optimizer.step()

        total_loss += loss.item() * len(yb)
        all_preds.extend(logits.argmax(dim=1).detach().cpu().numpy())
        all_targets.extend(yb.detach().cpu().numpy())

    avg_loss = total_loss / len(loader.dataset)
    avg_acc = accuracy_score(all_targets, all_preds)
    return avg_loss, avg_acc

EPOCHS = 60
PATIENCE = 8
best_val_loss = float("inf")
epochs_without_improvement = 0
best_state = None

history = {
    "train_loss": [], "val_loss": [],
    "train_acc": [], "val_acc": []
}

for epoch in range(EPOCHS):
    train_loss, train_acc = run_epoch(train_loader, training=True)
    val_loss, val_acc = run_epoch(val_loader, training=False)

    history["train_loss"].append(train_loss)
    history["val_loss"].append(val_loss)
    history["train_acc"].append(train_acc)
    history["val_acc"].append(val_acc)

    if val_loss < best_val_loss:
        best_val_loss = val_loss
        best_state = {k: v.detach().cpu().clone() for k, v in model.state_dict().items()}
        epochs_without_improvement = 0
    else:
        epochs_without_improvement += 1

    print(
        f"Epoch {epoch+1:02d}/{EPOCHS} | "
        f"train loss {train_loss:.4f}, acc {train_acc:.3f} | "
        f"val loss {val_loss:.4f}, acc {val_acc:.3f}"
    )

    if epochs_without_improvement >= PATIENCE:
        print(f"Early stopping triggered at epoch {epoch+1}.")
        break

if best_state is not None:
    model.load_state_dict(best_state)

## 8. Plot loss and accuracy convergence curves

In [ ]:
epochs_ran = range(1, len(history["train_loss"]) + 1)

plt.figure(figsize=(12, 4))

plt.subplot(1, 2, 1)
plt.plot(epochs_ran, history["train_loss"], label="Training loss")
plt.plot(epochs_ran, history["val_loss"], label="Validation loss")
plt.xlabel("Epoch")
plt.ylabel("Loss")
plt.title("Loss convergence")
plt.legend()
plt.grid(alpha=0.3)

plt.subplot(1, 2, 2)
plt.plot(epochs_ran, history["train_acc"], label="Training accuracy")
plt.plot(epochs_ran, history["val_acc"], label="Validation accuracy")
plt.xlabel("Epoch")
plt.ylabel("Accuracy")
plt.title("Accuracy convergence")
plt.legend()
plt.grid(alpha=0.3)

plt.tight_layout()
plt.show()

## 9. Evaluate on unseen test data

In [ ]:
model.eval()
with torch.no_grad():
    test_logits = model(torch.tensor(X_test, dtype=torch.float32).to(DEVICE))
    test_probabilities = torch.softmax(test_logits, dim=1).cpu().numpy()
    test_predictions = test_probabilities.argmax(axis=1)

print("Test accuracy:", round(accuracy_score(y_test, test_predictions), 4))
print("\\nClassification report:")
print(classification_report(
    y_test, test_predictions,
    labels=[0, 1],
    target_names=["negative", "positive"],
    zero_division=0
))

cm = confusion_matrix(y_test, test_predictions, labels=[0, 1])
ConfusionMatrixDisplay(
    confusion_matrix=cm, display_labels=["negative", "positive"]
).plot(cmap="Blues")
plt.title("Test confusion matrix")
plt.show()

## 10. Predict sentiment for new sample inputs

In [ ]:
def predict_sentiment(text):
    # Apply the already-fitted training vectorizer to unseen text.
    features = vectorizer.transform([text]).toarray().astype(np.float32)
    model.eval()
    with torch.no_grad():
        logits = model(torch.tensor(features, dtype=torch.float32).to(DEVICE))
        probabilities = torch.softmax(logits, dim=1).cpu().numpy()[0]

    predicted_id = int(np.argmax(probabilities))
    return {
        "text": text,
        "predicted_sentiment": id_to_label[predicted_id],
        "confidence": float(probabilities[predicted_id]),
        "negative_probability": float(probabilities[0]),
        "positive_probability": float(probabilities[1])
    }

sample_inputs = [
    "The product is excellent and I am very happy with it.",
    "The service was slow and the experience was disappointing.",
    "The app is easy to use and works really well."
]

for sample in sample_inputs:
    result = predict_sentiment(sample)
    print("\\nText:", result["text"])
    print("Prediction:", result["predicted_sentiment"])
    print("Confidence:", f'{result["confidence"]:.2%}')
    print(
        "Class probabilities:",
        f'negative={result["negative_probability"]:.2%},',
        f'positive={result["positive_probability"]:.2%}'
    )

## 11. Save the trained model and vectorizer

In [ ]:
# Save model weights and the TF-IDF vectorizer for later reuse.
torch.save(
    {
        "model_state_dict": model.state_dict(),
        "input_dim": input_dim,
        "label_to_id": label_to_id
    },
    "sentiment_model.pth"
)

import pickle
with open("tfidf_vectorizer.pkl", "wb") as file:
    pickle.dump(vectorizer, file)

print("Saved sentiment_model.pth and tfidf_vectorizer.pkl")

## 12. Conclusion

This project demonstrates a complete binary sentiment-classification pipeline. Text is transformed into TF-IDF features, then passed through a multi-layer neural network using Batch Normalization and Dropout. The model is trained with validation monitoring and early stopping, evaluated on a held-out test set, and used to predict unseen text with class probabilities.

**Important limitation:** The included dataset is intentionally small and suitable for demonstrating the required workflow, not for making reliable real-world sentiment decisions. For stronger performance, train with a larger, diverse, labelled dataset and report its source and limitations.